# Theory-Validation Simulation Experiments

This notebook numerically studies the theoretical claims in the paper **A Convolutional Framework for Detecting Event-Driven Dynamics in Energy Price Series**.

It contains two experiments:

1. **Numerical representation and approximation of statistical functionals.** Fixed, theory-aligned CNN/ReLU constructions are compared with the range, maximum drawup/down, slope-change, realized-volatility, and autoregressive statistics.
2. **Theorem 9 oracle pipeline across classical statistics.** Fixed slope-change, realized-volatility, and autoregressive comparator branches are evaluated separately and combined through a common trainable head across model-aligned and heterogeneous scenarios.

The experiments illustrate finite-sample implications of the theory; they do not replace the mathematical proofs.

## Theoretical map

- Theorems 1--3 give exact CNN representations for the range, drawup/down, and slope-change classifiers.
- Theorems 4--5 establish uniform ReLU-CNN approximation of realized volatility and the AR statistic on a bounded domain.
- Theorems 6--8 provide model-specific error bounds for the slope-change, realized-volatility, and AR comparators.
- Theorem 9 places those three fixed comparators in one common multi-branch CNN class and compares ERM with the best comparator plus a learning-complexity penalty.

Experiment 2 below is a restricted, computationally transparent instantiation of this pipeline rather than a numerical verification of the architecture-dependent VC bound. Its three statistic branches are fixed before the training sample is observed, and an affine common head is fitted to their normalized signed margins.

In [ ]:
# Imports and central configuration
from pathlib import Path
import math
import os
import random
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from IPython.display import display

from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


GLOBAL_SEED = 40
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(1)

# Experiment 1: representation and approximation (Sections 4.1--4.2).
BOUND_R = 1.0
REPRESENTATION_T_VALUES = [20, 40, 80]
APPROXIMATION_DEPTHS = [1, 2, 3, 4, 5, 6]
N_REPRESENTATION_WINDOWS = 1000

# Experiment 2: Theorem 9 three-comparator oracle pipeline (Section 4.3).
THEOREM9_T = 40
THEOREM9_SCENARIOS = ['slope', 'volatility', 'ar', 'mixed']
THEOREM9_TRAIN_SAMPLE_SIZES = [200, 500, 1000]
THEOREM9_N_TEST = 4000
THEOREM9_N_CALIBRATION = 20000

# Five repetitions provide a quick notebook run. Use 500 for the paper results.
THEOREM9_N_REPEATS = int(os.environ.get('THEOREM9_N_REPEATS', '5'))
THEOREM9_ALPHA = 0.05
THEOREM9_APPROXIMATION_DEPTH = 9
THEOREM9_CLIP_M = 1.0
THEOREM9_LOGISTIC_C = 1.0e4


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass


set_seed(GLOBAL_SEED)

_repository_candidates = [Path.cwd(), *Path.cwd().parents]
REPOSITORY_DIR = next(
    (path.resolve() for path in _repository_candidates if (path / 'code' / 'simulation').exists()),
    None,
)
if REPOSITORY_DIR is None:
    raise FileNotFoundError('Could not locate the repository root.')

CODE_DIR = REPOSITORY_DIR / 'code'
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from models.classical_statistic_classifiers import (
    ar_uniform_error_bound,
    build_classical_statistic_cnn,
    rv_uniform_error_bound,
)

SIMULATION_DIR = REPOSITORY_DIR / 'code' / 'simulation'

## Experiment 1: numerical representation and approximation

For a window (X=(X_1,\ldots,X_T)), define

\[
R(X)=\max_tX_t-\min_tX_t,
\quad
D^+(X)=\max_{s<t}(X_t-X_s),
\quad
D^-(X)=\max_{s<t}(X_s-X_t),
\]

\[
V(X)=\sum_{t=1}^{T-1}(X_{t+1}-X_t)^2,
\qquad
S_{\mathrm{AR}}(X)=\sum_{t=1}^{T-1}X_t(X_{t+1}-X_t).
\]

The fixed CNNs selected below are parameterised instances of the shared model framework and implement the constructive ideas in Theorems 1--3. For drawup and drawdown, the ReLU construction returns the positive part of the statistic; this is classifier-equivalent for every positive threshold. Realized volatility and the AR statistic use a deep tent-map ReLU approximation to the square function, local pairwise filters, and global sum pooling.

In [ ]:
# Direct statistical functionals
def range_statistic(x):
    return x.max(dim=1).values - x.min(dim=1).values


def maximum_drawup(x):
    running_min = torch.cummin(x, dim=1).values
    return (x - running_min).max(dim=1).values


def maximum_drawdown(x):
    running_max = torch.cummax(x, dim=1).values
    return (running_max - x).max(dim=1).values


def realized_volatility(x):
    returns = x[:, 1:] - x[:, :-1]
    return returns.square().sum(dim=1)


def ar_statistic(x):
    return (x[:, :-1] * (x[:, 1:] - x[:, :-1])).sum(dim=1)


def slope_contrast_coefficients(T, tau):
    # tau is the number of observations in the left segment.
    time = torch.arange(1, T + 1, dtype=torch.float64)
    left_t = time[:tau]
    right_t = time[tau:]
    left_centered = left_t - left_t.mean()
    right_centered = right_t - right_t.mean()
    coeff = torch.zeros(T, dtype=torch.float64)
    coeff[:tau] = left_centered / left_centered.square().sum()
    coeff[tau:] = -right_centered / right_centered.square().sum()
    return coeff


def slope_change_statistic(x):
    T = x.shape[1]
    contrasts = []
    for tau in range(3, T - 1):  # tau=3,...,T-2
        coeff = slope_contrast_coefficients(T, tau).to(x.device, x.dtype)
        contrasts.append(x @ coeff)
    return torch.stack(contrasts, dim=1).abs().max(dim=1).values


In [ ]:
# Theory-to-code architecture map
architecture_rows = []
for T in REPRESENTATION_T_VALUES:
    architecture_rows.extend([
        {
            'statistic': 'Range', 'T': T, 'code_construction': "build_classical_statistic_cnn('range')",
            'branches_or_filters': '1 branch, 2 filters', 'kernel_size': '1',
            'depth': '1 exact block', 'hidden/local channels': '2 / 2',
            'global_pooling': 'max',
        },
        {
            'statistic': 'Maximum drawup', 'T': T, 'code_construction': "build_classical_statistic_cnn('drawup')",
            'branches_or_filters': f'{T - 1} lag branches', 'kernel_size': f'2,...,{T}',
            'depth': '1 exact block per branch', 'hidden/local channels': '1 / 1 per branch',
            'global_pooling': 'max',
        },
        {
            'statistic': 'Maximum drawdown', 'T': T, 'code_construction': "build_classical_statistic_cnn('drawdown')",
            'branches_or_filters': f'{T - 1} lag branches', 'kernel_size': f'2,...,{T}',
            'depth': '1 exact block per branch', 'hidden/local channels': '1 / 1 per branch',
            'global_pooling': 'max',
        },
        {
            'statistic': 'Slope-change', 'T': T, 'code_construction': "build_classical_statistic_cnn('slope')",
            'branches_or_filters': f'{2 * (T - 4)} full-window filters', 'kernel_size': f'{T}',
            'depth': '1 exact block', 'hidden/local channels': f'{2 * (T - 4)} / 1',
            'global_pooling': 'max',
        },
        {
            'statistic': 'Realized volatility', 'T': T,
            'code_construction': "build_classical_statistic_cnn('realised_volatility')",
            'branches_or_filters': '1 local-difference branch', 'kernel_size': '2 then 1',
            'depth': f'square depth m in {APPROXIMATION_DEPTHS}',
            'hidden/local channels': 'constructive square subnet / 1',
            'global_pooling': 'sum',
        },
        {
            'statistic': 'AR statistic', 'T': T,
            'code_construction': "build_classical_statistic_cnn('ar')",
            'branches_or_filters': '1 pairwise branch', 'kernel_size': '2 then 1',
            'depth': f'square depth m in {APPROXIMATION_DEPTHS}',
            'hidden/local channels': 'three constructive square terms / 1',
            'global_pooling': 'sum',
        },
    ])

representation_architecture_df = pd.DataFrame(architecture_rows)
display(representation_architecture_df)


### Evaluation-window data-generating process

For each window length, the experiment generates 1,000 bounded AR-type trajectories with Gaussian innovations. The first 500 contain no change point. The remaining 500 contain one change point and are partitioned as evenly as possible among trend-change, volatility-change, and level-shift mechanisms; their mechanism labels are then randomly permuted. This design provides both stable and structurally perturbed paths without using the observations to train the fixed constructive networks.

In [ ]:
# Bounded evaluation windows from a structured time-series model with Gaussian errors
def generate_bounded_windows(n_windows, T, R=1.0, seed=40):
    rng = np.random.default_rng(seed)
    n_no_change = n_windows // 2
    n_change = n_windows - n_no_change
    paths = np.zeros((n_windows, T), dtype=np.float32)

    # The first half contains no change point. The second half is partitioned
    # approximately equally among trend, volatility, and level changes.
    change_modes = np.resize(np.arange(3), n_change)
    rng.shuffle(change_modes)

    for i in range(n_windows):
        phi = rng.uniform(0.3, 0.98)
        noise_scale = rng.uniform(0.04, 0.18)
        noise = rng.normal(0.0, noise_scale, size=T)
        path = np.zeros(T, dtype=np.float64)
        path[0] = noise[0]
        for t in range(1, T):
            path[t] = phi * path[t - 1] + noise[t]

        if i >= n_no_change:
            mode = change_modes[i - n_no_change]
            split = rng.integers(max(3, T // 4), min(T - 3, 3 * T // 4))
            if mode == 0:
                path[split:] += np.linspace(0.0, rng.uniform(-1.0, 1.0), T - split)
            elif mode == 1:
                path[split:] += rng.normal(0.0, noise_scale * 2.5, size=T - split)
            else:
                path[split:] += rng.uniform(-0.9, 0.9)

        scale = max(np.max(np.abs(path)), 1e-8)
        paths[i] = (0.95 * R * path / scale).astype(np.float32)

    return torch.tensor(paths)


In [ ]:
# Run Experiment 1
exact_rows = []
approximation_rows = []

for T in REPRESENTATION_T_VALUES:
    windows = generate_bounded_windows(
        N_REPRESENTATION_WINDOWS, T=T, R=BOUND_R, seed=GLOBAL_SEED + T
    )

    exact_pairs = {
        'Range': (build_classical_statistic_cnn('range', T), range_statistic(windows)),
        'Maximum drawup': (build_classical_statistic_cnn('drawup', T), maximum_drawup(windows)),
        'Maximum drawdown': (build_classical_statistic_cnn('drawdown', T), maximum_drawdown(windows)),
        'Slope-change': (build_classical_statistic_cnn('slope', T), slope_change_statistic(windows)),
    }
    for statistic_name, (model, target) in exact_pairs.items():
        with torch.no_grad():
            estimate = model(windows)
        absolute_error = torch.abs(estimate - target)
        threshold = torch.quantile(target, 0.5)
        agreement = ((estimate > threshold) == (target > threshold)).float().mean()
        exact_rows.append({
            'statistic': statistic_name,
            'T': T,
            'max_abs_error': float(absolute_error.max()),
            'mean_abs_error': float(absolute_error.mean()),
            'classification_agreement': float(agreement),
        })

    approximation_targets = {
        'Realized volatility': realized_volatility(windows),
        'AR statistic': ar_statistic(windows),
    }
    for statistic_name, target in approximation_targets.items():
        threshold = torch.quantile(target, 0.5)
        for depth in APPROXIMATION_DEPTHS:
            if statistic_name == 'Realized volatility':
                model = build_classical_statistic_cnn(
                    'realised_volatility', T, BOUND_R, depth
                )
                error_bound = rv_uniform_error_bound(T, BOUND_R, depth)
            else:
                model = build_classical_statistic_cnn('ar', T, BOUND_R, depth)
                error_bound = ar_uniform_error_bound(T, BOUND_R, depth)

            with torch.no_grad():
                estimate = model(windows)
            absolute_error = torch.abs(estimate - target)
            target_label = target > threshold
            estimated_label = estimate > threshold
            disagreement = (estimated_label != target_label).float()
            outside_margin = torch.abs(target - threshold) > error_bound
            margin_disagreement = (
                float(disagreement[outside_margin].mean()) if outside_margin.any() else np.nan
            )
            approximation_rows.append({
                'statistic': statistic_name,
                'T': T,
                'depth': depth,
                'max_abs_error': float(absolute_error.max()),
                'mean_abs_error': float(absolute_error.mean()),
                'theoretical_error_bound': float(error_bound),
                'bound_satisfied_on_sample': bool(float(absolute_error.max()) <= error_bound + 1e-6),
                'classification_agreement': float(1.0 - disagreement.mean()),
                'outside_margin_n': int(outside_margin.sum()),
                'outside_margin_disagreement': margin_disagreement,
            })

exact_representation_df = pd.DataFrame(exact_rows)
approximation_df = pd.DataFrame(approximation_rows)

print('Exact representation results')
display(exact_representation_df)
print('\nApproximation results (first rows)')
display(approximation_df.head(12))
print('\nAll sampled approximation errors satisfy the analytic bounds:',
      approximation_df['bound_satisfied_on_sample'].all())
print('Maximum disagreement outside the theoretical margin:',
      approximation_df['outside_margin_disagreement'].fillna(0).max())


In [ ]:
# Plot Experiment 1 approximation errors
from matplotlib.lines import Line2D

PLOT_FONT_SIZE = 12

# T is encoded by marker shape. Line style distinguishes the observed error
# from the analytic bound, so the figure remains readable in grayscale.
t_styles = {
    20: {'marker': 'o'},
    40: {'marker': 's'},
    80: {'marker': '^'},
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=False)
for ax, statistic_name in zip(axes, ['Realized volatility', 'AR statistic']):
    subset = approximation_df[approximation_df['statistic'] == statistic_name]
    for T, group in subset.groupby('T'):
        group = group.sort_values('depth')
        style = t_styles[T]
        observed_line, = ax.plot(
            group['depth'], group['max_abs_error'],
            linestyle='-', marker=style['marker'],
            linewidth=2.0, markersize=6,
        )
        ax.plot(
            group['depth'], group['theoretical_error_bound'],
            color=observed_line.get_color(),
            linestyle='--', marker=style['marker'],
            linewidth=1.0, markersize=6, markerfacecolor='white',
            markeredgewidth=1.2, alpha=0.85,
        )
    ax.set_yscale('log')
    ax.set_xlabel('Approximation level m', fontsize=PLOT_FONT_SIZE)
    ax.set_ylabel('Maximum absolute error', fontsize=PLOT_FONT_SIZE)
    ax.set_title(statistic_name, fontsize=PLOT_FONT_SIZE)
    ax.tick_params(axis='both', which='both', labelsize=PLOT_FONT_SIZE)
    ax.grid(alpha=0.25)

t_handles = [
    Line2D([0], [0], color='black', linestyle='None',
           marker=style['marker'], label=f'T={T}')
    for T, style in t_styles.items()
]
meaning_handles = [
    Line2D([0], [0], color='black', linewidth=2.0, linestyle='-',
           label='Observed maximum error'),
    Line2D([0], [0], color='black', linewidth=1.2, linestyle='--',
           label='Analytic upper bound'),
]
fig.legend(
    handles=t_handles + meaning_handles,
    loc='upper center', bbox_to_anchor=(0.5, 0.90), ncol=5,
    frameon=False, fontsize=PLOT_FONT_SIZE,
)
# fig.suptitle('Approximation error versus ReLU depth', fontsize=PLOT_FONT_SIZE, y=0.99)
fig.subplots_adjust(left=0.08, right=0.98, bottom=0.15, top=0.76, wspace=0.20)
plt.show()

agreement_pivot = approximation_df.pivot_table(
    index=['statistic', 'T'], columns='depth', values='classification_agreement'
)
print('Classification agreement with the exact statistic')
display(agreement_pivot)


## Experiment 2: Theorem 9 oracle pipeline across classical statistics

Theorem 9 compares ERM over a common multi-branch CNN class with the best of three fully specified deterministic comparators: slope change, realized volatility, and autoregressive explosiveness. The experiment uses three fixed statistic branches and a common trainable head.

Four scenarios are considered.

1. **Slope.** Gaussian piecewise-linear paths contain a label-independent nuisance trend and, under the alternative, a continuous peak/valley slope break.
2. **Volatility.** Bounded independent returns have a probability of a nonzero move that changes by class.
3. **AR.** Gaussian AR paths use Φ=1 under the null and Φ=1.10 under the alternative. A paired 100-repetition sensitivity pilot compared Φ in {1.08, 1.10, 1.12}. The value 1.10 was the smallest choice that materially reduced AR error without making the task nearly trivial.
4. **Mixed.** All null trajectories follow the same Gaussian random walk. Under the alternative, a slope break, additional volatility jumps, or an explosive AR recursion is selected with equal probability. Heterogeneity therefore arises only under the alternative, while the definition of a normal trajectory remains unchanged. The subtype-specific signal parameters are chosen for this heterogeneous experiment and are not calibrated to reproduce the three standalone DGPs. The mixed scenario is therefore not a direct mixture of the standalone DGPs.

For each scenario, a separate null-only calibration sample is generated before any training sample. It fixes the three thresholds at null quantile 1-α, the positive feature scales, the clipping radius, and the RV/AR approximation networks. Conditional on this external calibration, all comparator specifications are fixed before training, as required by Theorem 9.

The slope branch receives the un-clipped input. The RV and AR branches receive componentwise clipped inputs and use the fixed triangular-map approximations at level m=9 from Experiment 1. We report the following quantities.

- each fixed comparator's test error.
- the empirical best-comparator error.
- exact training-set 0--1 ERM over the three fixed comparators.
- a practical affine common head fitted by logistic cross-entropy.

The empirical best comparator is an infeasible test-set oracle. The single-comparator ERM selects one fixed rule by training error, while the affine head combines the three normalized statistic margins using logistic cross-entropy. A negative difference between the head's test error and the oracle error means that the learned combination outperforms every single fixed rule.

Training samples are nested within each repetition, and the logistic solver is run to convergence. Only the training sample size varies. The approximation level and all quantities fixed by calibration are held constant.

In [ ]:
# Theorem 9 model-aligned and heterogeneous data-generating processes
COMPARATOR_NAMES = ['slope', 'volatility', 'ar']

SLOPE_SIGMA = 0.01
SLOPE_KAPPA = 0.03
SLOPE_BASELINE_MAX = 0.008

VOLATILITY_ACTIVE_PROB_NULL = 0.20
VOLATILITY_ACTIVE_PROB_ALT = 0.55

AR_PHI_NULL = 1.0
AR_PHI_ALT = 1.10
AR_SIGMA = 0.03
AR_SCALE_MULTIPLIER = 4.0

MIXED_SIGMA = 0.01
MIXED_SLOPE_KAPPA = 0.02
MIXED_VOLATILITY_JUMP = 0.04
MIXED_VOLATILITY_ACTIVE_PROB = 0.40
MIXED_AR_PHI = AR_PHI_ALT

def _generate_slope_family(labels, T, rng):
    n = len(labels)
    time_index = np.arange(1, T + 1, dtype=np.float64)
    intercept = rng.uniform(-0.05, 0.05, size=n)
    baseline_slope = rng.uniform(
        -SLOPE_BASELINE_MAX, SLOPE_BASELINE_MAX, size=n
    )
    paths = (
        intercept[:, None]
        + baseline_slope[:, None] * time_index[None, :]
        + rng.normal(0.0, SLOPE_SIGMA, size=(n, T))
    )

    event_indices = np.flatnonzero(labels == 1)
    if len(event_indices):
        split_counts = rng.integers(
            T // 2 - 2, T // 2 + 3, size=len(event_indices)
        )
        directions = rng.choice([-1.0, 1.0], size=len(event_indices))
        for row_index, split_count, direction in zip(
            event_indices, split_counts, directions
        ):
            # A continuous peak/valley has slopes +kappa/2 and -kappa/2.
            # It preserves the required slope separation without introducing
            # a large one-sided terminal level that would favour the AR score.
            centered_time = time_index - split_count
            paths[row_index] += (
                -0.5 * direction * SLOPE_KAPPA * np.abs(centered_time)
            )
    return paths


def _generate_volatility_family(labels, T, rng):
    n = len(labels)
    step_bound = 1.0 / (T - 1)
    active_probabilities = np.where(
        labels == 1,
        VOLATILITY_ACTIVE_PROB_ALT,
        VOLATILITY_ACTIVE_PROB_NULL,
    )
    active = rng.random((n, T - 1)) < active_probabilities[:, None]
    signs = rng.choice([-1.0, 1.0], size=(n, T - 1))
    returns = step_bound * active * signs
    paths = np.zeros((n, T), dtype=np.float64)
    paths[:, 1:] = np.cumsum(returns, axis=1)
    return paths


def _ar_scale(T):
    variance_multiplier = sum(AR_PHI_ALT ** (2 * j) for j in range(T))
    return AR_SCALE_MULTIPLIER * AR_SIGMA * math.sqrt(variance_multiplier)


def _generate_ar_family(labels, T, rng):
    n = len(labels)
    phi = np.where(labels == 1, AR_PHI_ALT, AR_PHI_NULL)
    innovations = rng.normal(0.0, AR_SIGMA, size=(n, T))
    paths = np.zeros((n, T), dtype=np.float64)
    paths[:, 0] = innovations[:, 0]
    for t in range(1, T):
        paths[:, t] = phi * paths[:, t - 1] + innovations[:, t]
    return paths / _ar_scale(T)


def _generate_mixed_family(labels, T, rng):
    """Common null process with three heterogeneous event mechanisms."""
    n = len(labels)
    time_index = np.arange(1, T + 1, dtype=np.float64)
    innovations = rng.normal(0.0, MIXED_SIGMA, size=(n, T))
    paths = np.cumsum(innovations, axis=1)
    family_codes = rng.integers(0, len(COMPARATOR_NAMES), size=n)
    families = np.asarray(COMPARATOR_NAMES, dtype=object)[family_codes]

    slope_indices = np.flatnonzero((labels == 1) & (families == 'slope'))
    if len(slope_indices):
        split_counts = rng.integers(
            T // 2 - 2, T // 2 + 3, size=len(slope_indices)
        )
        directions = rng.choice([-1.0, 1.0], size=len(slope_indices))
        for row_index, split_count, direction in zip(
            slope_indices, split_counts, directions
        ):
            paths[row_index] += (
                -0.5 * direction * MIXED_SLOPE_KAPPA
                * np.abs(time_index - split_count)
            )

    volatility_indices = np.flatnonzero(
        (labels == 1) & (families == 'volatility')
    )
    if len(volatility_indices):
        active = (
            rng.random((len(volatility_indices), T - 1))
            < MIXED_VOLATILITY_ACTIVE_PROB
        )
        signs = rng.choice(
            [-1.0, 1.0], size=(len(volatility_indices), T - 1)
        )
        jumps = MIXED_VOLATILITY_JUMP * active * signs
        paths[volatility_indices, 1:] += np.cumsum(jumps, axis=1)

    ar_indices = np.flatnonzero((labels == 1) & (families == 'ar'))
    if len(ar_indices):
        ar_paths = np.zeros((len(ar_indices), T), dtype=np.float64)
        ar_innovations = innovations[ar_indices]
        ar_paths[:, 0] = ar_innovations[:, 0]
        for t in range(1, T):
            ar_paths[:, t] = (
                MIXED_AR_PHI * ar_paths[:, t - 1] + ar_innovations[:, t]
            )
        paths[ar_indices] = ar_paths

    return paths, families


def generate_theorem9_sample(n_samples, T, scenario, seed, force_label=None):
    if scenario not in THEOREM9_SCENARIOS:
        raise ValueError(f'Unknown scenario: {scenario}')
    rng = np.random.default_rng(seed)

    if force_label is None:
        n_event = n_samples // 2
        labels = np.concatenate([
            np.zeros(n_samples - n_event, dtype=np.int64),
            np.ones(n_event, dtype=np.int64),
        ])
        rng.shuffle(labels)
    else:
        labels = np.full(n_samples, int(force_label), dtype=np.int64)

    if scenario == 'mixed':
        paths, families = _generate_mixed_family(labels, T, rng)
    else:
        families = np.full(n_samples, scenario, dtype=object)
        generator = {
            'slope': _generate_slope_family,
            'volatility': _generate_volatility_family,
            'ar': _generate_ar_family,
        }[scenario]
        paths = generator(labels, T, rng)

    return paths.astype(np.float32), labels, families


def balanced_nested_subset(x_pool, y_pool, n_samples, seed):
    if n_samples % 2:
        raise ValueError('Training sample sizes must be even.')
    half = n_samples // 2
    selected = np.concatenate([
        np.flatnonzero(y_pool == 0)[:half],
        np.flatnonzero(y_pool == 1)[:half],
    ])
    rng = np.random.default_rng(seed)
    rng.shuffle(selected)
    return x_pool[selected], y_pool[selected]

In [ ]:
# Fixed comparator branches, calibration, and common-head estimators
THEOREM9_VOLATILITY_CNN = build_classical_statistic_cnn(
    'realised_volatility', THEOREM9_T,
    THEOREM9_CLIP_M, THEOREM9_APPROXIMATION_DEPTH,
)
THEOREM9_AR_CNN = build_classical_statistic_cnn(
    'ar', THEOREM9_T, THEOREM9_CLIP_M, THEOREM9_APPROXIMATION_DEPTH,
)


def score_theorem9_branches(x, include_diagnostics=False):
    x_tensor = torch.as_tensor(x, dtype=torch.float32)
    clipped = torch.clamp(
        x_tensor, min=-THEOREM9_CLIP_M, max=THEOREM9_CLIP_M
    )
    with torch.no_grad():
        slope_exact = slope_change_statistic(x_tensor).cpu().numpy()
        volatility_approx = THEOREM9_VOLATILITY_CNN(clipped).cpu().numpy()
        ar_approx = THEOREM9_AR_CNN(clipped).cpu().numpy()

    scores = {
        'slope_exact': slope_exact,
        'volatility_approx': volatility_approx,
        'ar_approx': ar_approx,
    }
    if include_diagnostics:
        scores.update({
            'volatility_exact': realized_volatility(clipped).cpu().numpy(),
            'ar_exact': ar_statistic(clipped).cpu().numpy(),
            'outside_clip': np.max(np.abs(x), axis=1) > THEOREM9_CLIP_M,
        })
    return scores


def calibrate_theorem9_comparators(scenario, seed):
    x_null, _, _ = generate_theorem9_sample(
        THEOREM9_N_CALIBRATION,
        THEOREM9_T,
        scenario,
        seed,
        force_label=0,
    )
    scores = score_theorem9_branches(x_null, include_diagnostics=True)
    epsilon_v = rv_uniform_error_bound(
        THEOREM9_T, THEOREM9_CLIP_M, THEOREM9_APPROXIMATION_DEPTH
    )
    epsilon_ar = ar_uniform_error_bound(
        THEOREM9_T, THEOREM9_CLIP_M, THEOREM9_APPROXIMATION_DEPTH
    )

    thresholds = {
        'slope': float(np.quantile(
            scores['slope_exact'], 1.0 - THEOREM9_ALPHA, method='higher'
        )),
        'volatility': float(np.quantile(
            scores['volatility_exact'], 1.0 - THEOREM9_ALPHA, method='higher'
        )),
        'ar': float(np.quantile(
            scores['ar_exact'], 1.0 - THEOREM9_ALPHA, method='higher'
        )),
    }

    raw_margins = np.column_stack([
        scores['slope_exact'] - thresholds['slope'],
        scores['volatility_approx'] - (thresholds['volatility'] + epsilon_v),
        scores['ar_approx'] - (thresholds['ar'] + epsilon_ar),
    ])
    feature_scales = np.maximum(raw_margins.std(axis=0, ddof=1), 1.0e-8)
    calibration_predictions = raw_margins > 0.0

    max_rv_error = float(np.max(np.abs(
        scores['volatility_approx'] - scores['volatility_exact']
    )))
    max_ar_error = float(np.max(np.abs(
        scores['ar_approx'] - scores['ar_exact']
    )))
    if max_rv_error > epsilon_v + 1.0e-6:
        raise RuntimeError('RV approximation exceeded its analytic bound.')
    if max_ar_error > epsilon_ar + 1.0e-6:
        raise RuntimeError('AR approximation exceeded its analytic bound.')

    return {
        'scenario': scenario,
        'thresholds': thresholds,
        'epsilon_v': epsilon_v,
        'epsilon_ar': epsilon_ar,
        'feature_scales': feature_scales,
        'calibration_false_positive_rates': calibration_predictions.mean(axis=0),
        'calibration_tail_rate': float(scores['outside_clip'].mean()),
        'max_rv_approximation_error': max_rv_error,
        'max_ar_approximation_error': max_ar_error,
    }


def theorem9_features_and_predictions(x, calibration):
    scores = score_theorem9_branches(x)
    thresholds = calibration['thresholds']
    raw_margins = np.column_stack([
        scores['slope_exact'] - thresholds['slope'],
        scores['volatility_approx']
        - (thresholds['volatility'] + calibration['epsilon_v']),
        scores['ar_approx'] - (thresholds['ar'] + calibration['epsilon_ar']),
    ])
    normalized_features = raw_margins / calibration['feature_scales'][None, :]
    comparator_predictions = (raw_margins > 0.0).astype(np.int64)
    return normalized_features, comparator_predictions


def error_rate(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) != np.asarray(y_pred)))


def fit_common_head(x_train, y_train):
    model = LogisticRegression(
        C=THEOREM9_LOGISTIC_C,
        solver='lbfgs',
        max_iter=2000,
        random_state=GLOBAL_SEED,
    )
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', ConvergenceWarning)
        model.fit(x_train, y_train)
    return model

In [ ]:
# Run the Theorem 9 oracle-pipeline experiment
start_time = time.time()
theorem9_rows = []
calibrations = {}
for scenario_index, scenario in enumerate(THEOREM9_SCENARIOS):
    calibration_seed = GLOBAL_SEED + 1_000_000 + 10_000 * scenario_index
    calibrations[scenario] = calibrate_theorem9_comparators(
        scenario, calibration_seed
    )

    for repeat in range(THEOREM9_N_REPEATS):
        base_seed = GLOBAL_SEED + 10_000_000 * scenario_index + 10_000 * repeat
        x_train_pool, y_train_pool, _ = generate_theorem9_sample(
            max(THEOREM9_TRAIN_SAMPLE_SIZES),
            THEOREM9_T,
            scenario,
            seed=base_seed + 1,
        )
        x_test, y_test, _ = generate_theorem9_sample(
            THEOREM9_N_TEST,
            THEOREM9_T,
            scenario,
            seed=base_seed + 2,
        )

        test_features, test_comparator_predictions = (
            theorem9_features_and_predictions(x_test, calibrations[scenario])
        )
        comparator_test_errors = np.array([
            error_rate(y_test, test_comparator_predictions[:, index])
            for index in range(len(COMPARATOR_NAMES))
        ])
        oracle_index = int(np.argmin(comparator_test_errors))
        oracle_test_error = float(comparator_test_errors[oracle_index])

        for n_train in THEOREM9_TRAIN_SAMPLE_SIZES:
            x_train, y_train = balanced_nested_subset(
                x_train_pool,
                y_train_pool,
                n_train,
                seed=base_seed + n_train,
            )
            train_features, train_comparator_predictions = (
                theorem9_features_and_predictions(
                    x_train, calibrations[scenario]
                )
            )
            comparator_train_errors = np.array([
                error_rate(y_train, train_comparator_predictions[:, index])
                for index in range(len(COMPARATOR_NAMES))
            ])

            # Exact empirical 0--1 minimization over the three fixed comparators.
            selector_index = int(np.argmin(comparator_train_errors))
            selector_train_error = float(comparator_train_errors[selector_index])
            selector_test_error = float(comparator_test_errors[selector_index])

            # Practical affine common head fitted with the logistic surrogate.
            common_head = fit_common_head(train_features, y_train)
            common_train_prediction = common_head.predict(train_features)
            common_test_prediction = common_head.predict(test_features)
            common_train_probability = common_head.predict_proba(train_features)[:, 1]
            common_test_probability = common_head.predict_proba(test_features)[:, 1]
            common_train_error = error_rate(y_train, common_train_prediction)
            common_test_error = error_rate(y_test, common_test_prediction)

            theorem9_rows.append({
                'scenario': scenario,
                'ar_phi_alt': AR_PHI_ALT,
                'mixed_ar_phi': MIXED_AR_PHI,
                'mixed_family_probability': 1.0 / len(COMPARATOR_NAMES),
                'repeat': repeat,
                'n_train': n_train,
                'slope_comparator_train_error': comparator_train_errors[0],
                'volatility_comparator_train_error': comparator_train_errors[1],
                'ar_comparator_train_error': comparator_train_errors[2],
                'slope_comparator_test_error': comparator_test_errors[0],
                'volatility_comparator_test_error': comparator_test_errors[1],
                'ar_comparator_test_error': comparator_test_errors[2],
                'oracle_comparator': COMPARATOR_NAMES[oracle_index],
                'oracle_comparator_test_error': oracle_test_error,
                'selector_comparator': COMPARATOR_NAMES[selector_index],
                'selector_train_error': selector_train_error,
                'selector_test_error': selector_test_error,
                'selector_oracle_excess': selector_test_error - oracle_test_error,
                'common_head_train_error': common_train_error,
                'common_head_test_error': common_test_error,
                'common_head_generalization_gap': (
                    common_test_error - common_train_error
                ),
                'common_head_abs_generalization_gap': abs(
                    common_test_error - common_train_error
                ),
                'common_head_oracle_excess': common_test_error - oracle_test_error,
                'common_head_train_auc': roc_auc_score(
                    y_train, common_train_probability
                ),
                'common_head_test_auc': roc_auc_score(
                    y_test, common_test_probability
                ),
            })

        if (
            (repeat + 1) % max(1, THEOREM9_N_REPEATS // 10) == 0
            or repeat + 1 == THEOREM9_N_REPEATS
        ):
            print(
                f'{scenario}: completed {repeat + 1}/'
                f'{THEOREM9_N_REPEATS} repetitions'
            )

theorem9_results_df = pd.DataFrame(theorem9_rows)
print(f'Experiment runtime: {time.time() - start_time:.1f} seconds')
display(theorem9_results_df.head())

In [ ]:
# Aggregate the Theorem 9 experiment and create paper-ready diagnostics
summary_metrics = [
    'slope_comparator_test_error',
    'volatility_comparator_test_error',
    'ar_comparator_test_error',
    'oracle_comparator_test_error',
    'selector_train_error',
    'selector_test_error',
    'selector_oracle_excess',
    'common_head_train_error',
    'common_head_test_error',
    'common_head_generalization_gap',
    'common_head_abs_generalization_gap',
    'common_head_oracle_excess',
    'common_head_test_auc',
]

summary_rows = []
for (scenario, n_train), group in theorem9_results_df.groupby(
    ['scenario', 'n_train'], sort=False
):
    row = {'scenario': scenario, 'n_train': n_train}
    for metric in summary_metrics:
        row[f'{metric}_mean'] = group[metric].mean()
        row[f'{metric}_std'] = group[metric].std(ddof=1)
    summary_rows.append(row)

theorem9_summary_df = pd.DataFrame(summary_rows).sort_values(
    ['scenario', 'n_train']
)
calibration_rows = []
for scenario, calibration in calibrations.items():
    calibration_rows.append({
        'scenario': scenario,
        'alpha': THEOREM9_ALPHA,
        'ar_phi_alt': AR_PHI_ALT,
        'mixed_ar_phi': MIXED_AR_PHI,
        'mixed_family_probability': 1.0 / len(COMPARATOR_NAMES),
        'slope_threshold': calibration['thresholds']['slope'],
        'volatility_threshold': calibration['thresholds']['volatility'],
        'ar_threshold': calibration['thresholds']['ar'],
        'epsilon_v': calibration['epsilon_v'],
        'epsilon_ar': calibration['epsilon_ar'],
        'slope_calibration_fpr': (
            calibration['calibration_false_positive_rates'][0]
        ),
        'volatility_calibration_fpr': (
            calibration['calibration_false_positive_rates'][1]
        ),
        'ar_calibration_fpr': (
            calibration['calibration_false_positive_rates'][2]
        ),
        'calibration_tail_rate': calibration['calibration_tail_rate'],
        'max_rv_approximation_error': (
            calibration['max_rv_approximation_error']
        ),
        'max_ar_approximation_error': (
            calibration['max_ar_approximation_error']
        ),
    })
theorem9_calibration_df = pd.DataFrame(calibration_rows)
oracle_frequency_source = theorem9_results_df.drop_duplicates(
    ['scenario', 'repeat']
)
theorem9_oracle_frequency_df = (
    oracle_frequency_source.groupby(['scenario', 'oracle_comparator'])
    .size()
    .rename('count')
    .reset_index()
)
theorem9_oracle_frequency_df['frequency'] = (
    theorem9_oracle_frequency_df['count']
    / theorem9_oracle_frequency_df.groupby('scenario')['count'].transform('sum')
)
scenario_titles = {
    'slope': 'Slope-change scenario',
    'volatility': 'Volatility-change scenario',
    'ar': 'AR-explosive scenario',
    'mixed': 'Heterogeneous mixture',
}
method_styles = {
    'oracle': {
        'label': 'Oracle best comparator',
        'color': '#1f77b4',
        'linestyle': '--',
    },
    'selector': {
        'label': 'Single-comparator ERM',
        'color': '#ff7f0e',
        'linestyle': '-.',
    },
    'ours': {
        'label': 'Joint affine-head classifier',
        'color': '#2ca02c',
        'linestyle': '-',
    },
}

fig, axes = plt.subplots(
    2, 2, figsize=(12, 8), sharey=True, constrained_layout=True
)
for ax, scenario in zip(axes.flat, THEOREM9_SCENARIOS):
    subset = theorem9_summary_df[
        theorem9_summary_df['scenario'] == scenario
    ]
    oracle_style = method_styles['oracle']
    ax.plot(
        subset['n_train'],
        np.zeros(len(subset)),
        marker='o',
        linestyle=oracle_style['linestyle'],
        color=oracle_style['color'],
        label=oracle_style['label'],
    )
    for metric, method in [
        ('selector_oracle_excess', 'selector'),
        ('common_head_oracle_excess', 'ours'),
    ]:
        style = method_styles[method]
        ax.errorbar(
            subset['n_train'],
            subset[f'{metric}_mean'],
            yerr=subset[f'{metric}_std'],
            marker='o',
            linestyle=style['linestyle'],
            color=style['color'],
            capsize=3,
            label=style['label'],
        )
    ax.set_xscale('log')
    ax.set_title(scenario_titles[scenario])
    ax.set_xlabel('Training sample size N')
    ax.set_ylabel(
        'Test-error difference from oracle benchmark\n(lower is better)'
    )
    ax.grid(alpha=0.25)
axes[0, 0].legend(fontsize=9)
plt.show()

display_columns = [
    'scenario',
    'n_train',
    'oracle_comparator_test_error_mean',
    'selector_test_error_mean',
    'common_head_test_error_mean',
    'common_head_abs_generalization_gap_mean',
    'common_head_oracle_excess_mean',
]
display(theorem9_summary_df[display_columns])
display(theorem9_calibration_df)
display(theorem9_oracle_frequency_df)
